In [1]:
# Practical Assignment 5: Ensemble Learning and Performance Evaluation
# Dataset: Breast Cancer Wisconsin Diagnostic (built into scikit-learn)

import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import StackingClassifier, AdaBoostClassifier, VotingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

# 1. Load dataset
dataset = load_breast_cancer()
X = dataset.data
y = dataset.target
print("Dataset shape:", X.shape)
print("Target classes:", dataset.target_names)  # 0=malignant, 1=benign

# 2. Split data (stratified to preserve class proportions)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 3. Define base classifiers
base_estimators = [
    ("Logistic Regression", make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, random_state=42)
    )),
    ("Decision Tree", DecisionTreeClassifier(
        max_depth=5, random_state=42
    )),
    ("KNN", make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=5)
    )),
    ("SVM", make_pipeline(
        StandardScaler(),
        SVC(kernel="rbf", C=1.0, random_state=42)
    ))
]

# 4. Define ensemble classifiers
stacking_model = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(max_iter=2000),
    cv=5
)

# AdaBoost uses decision-tree stumps as weak learners.
# For scikit-learn versions before 1.2, replace estimator= with base_estimator=.
adaboost_model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
    n_estimators=100,
    learning_rate=0.5,
    random_state=42
)

voting_model = VotingClassifier(
    estimators=base_estimators,
    voting="hard"  # majority voting
)

# 5. Add base and ensemble models to one comparison dictionary
models = {name: model for name, model in base_estimators}
models["Stacking"] = stacking_model
models["AdaBoost"] = adaboost_model
models["Hard Voting"] = voting_model

# 6. Train, predict, and evaluate each model
results = []
for model_name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-score": f1_score(y_test, y_pred)
    })

    print("\\n", model_name)
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(
        y_test, y_pred, target_names=dataset.target_names
    ))

# 7. Display comparative performance table
results_df = pd.DataFrame(results)
print("\\nPerformance Comparison:")
print(results_df.to_string(index=False))

# Optional: save the results for the practical report
results_df.to_csv("ensemble_performance_results.csv", index=False)


Dataset shape: (569, 30)
Target classes: ['malignant' 'benign']
\n Logistic Regression
[[41  1]
 [ 1 71]]
              precision    recall  f1-score   support

   malignant       0.98      0.98      0.98        42
      benign       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114

\n Decision Tree
[[39  3]
 [ 6 66]]
              precision    recall  f1-score   support

   malignant       0.87      0.93      0.90        42
      benign       0.96      0.92      0.94        72

    accuracy                           0.92       114
   macro avg       0.91      0.92      0.92       114
weighted avg       0.92      0.92      0.92       114

\n KNN
[[39  3]
 [ 2 70]]
              precision    recall  f1-score   support

   malignant       0.95      0.93      0.94        42
      benign       0.96      0.97      0.97        72

    accuracy  